In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, time
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")

def newest(pattern, label):
    paths = list(INPUT.rglob(pattern))
    if not paths:
        raise FileNotFoundError(f"Attach the saved output containing {label}: {pattern}")
    return max(paths, key=lambda p: p.stat().st_mtime)

metadata_paths = list(INPUT.rglob("simulation_metadata.json"))
binary_dirs = []
for mp in metadata_paths:
    if mp.parent.name != "binary":
        continue
    try:
        meta = json.loads(mp.read_text(encoding="utf-8"))
    except Exception:
        continue
    if meta.get("family") == "binary" and int(meta.get("parameter_draws_unique", 0)) >= 100_000:
        binary_dirs.append((meta.get("created_utc", ""), mp.parent))
if not binary_dirs:
    raise FileNotFoundError("Attach Day 7 Cosmic swan Bayesian output with binary simulation arrays.")
_, BINARY_DIR = sorted(binary_dirs, key=lambda x: x[0])[-1]
observations = np.load(BINARY_DIR / "observations.npy", mmap_mode="r")
parameters = pd.read_csv(BINARY_DIR / "parameters.csv")
time_days = np.asarray(np.load(BINARY_DIR / "time_days.npy"), dtype=float)
if observations.shape != (len(parameters), len(time_days)):
    raise ValueError("Day 7 observations, parameters, and time axis have inconsistent shapes.")

ALIAS_PATH = newest("day9_3_binary_alias_candidate_scores.csv", "Day 9.3 alias candidate scores")
NUTS_PATH = newest("day9_9_nuts_case_diagnostics.csv", "Day 9.9 NUTS diagnostics")
MAP_PATH = newest("day9_9_map_case_summary.csv", "Day 9.9 MAP case summary")
MAP_PARAMS_PATH = newest("day9_9_map_parameter_recovery.csv", "Day 9.9 MAP parameter estimates")
aliases = pd.read_csv(ALIAS_PATH)
nuts = pd.read_csv(NUTS_PATH)
map_cases = pd.read_csv(MAP_PATH)
map_params = pd.read_csv(MAP_PARAMS_PATH)
required_alias = {"simulation_row", "bic", "fit_period_days", "phase", "duration_days", "noise_sigma"}
if not required_alias.issubset(aliases.columns):
    raise ValueError(f"Day 9.3 alias CSV missing: {sorted(required_alias-set(aliases.columns))}")

review_rows = set(nuts.loc[nuts.status.astype(str).ne("screen_pass"), "simulation_row"].astype(int))
if "period_recovered_within_tolerance" in map_cases.columns:
    map_miss_rows = set(map_cases.loc[~map_cases.period_recovered_within_tolerance.astype(bool), "simulation_row"].astype(int))
else:
    map_miss_rows = set()
CASE_ROWS = sorted(review_rows | map_miss_rows)
if not CASE_ROWS:
    raise ValueError("No Day 9.9 NUTS review rows or MAP misses were found.")
if not set(CASE_ROWS).issubset(set(aliases.simulation_row.astype(int))):
    raise ValueError("Day 9.3 alias CSV does not include all selected rows.")
if max(CASE_ROWS) >= len(parameters):
    raise ValueError("A selected simulation row is outside the Day 7 table.")

OUT = WORK / ("day9-10-period-profile-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
OUT.mkdir(parents=True, exist_ok=False)
print("[PASS] Day 7:", BINARY_DIR)
print("[PASS] Day 9.3:", ALIAS_PATH)
print("[PASS] Day 9.9 NUTS:", NUTS_PATH)
print("Selected profile rows:", CASE_ROWS)
print("Output:", OUT)


In [ ]:
def smooth_clip_np(x, sharpness=500.0):
    return (np.logaddexp(0.0, sharpness*x) - np.logaddexp(0.0, sharpness*(x-1.0))) / sharpness

def predict_np(t, baseline, period, phase, depth, ratio, duration, ellip, reflection):
    orb = np.mod(t/period-phase+0.5, 1.0)-0.5
    sec = np.mod(orb, 1.0)-0.5
    half = max(float(duration)/2.0, 1e-6)
    ingress = max(half*0.18, 1e-6)
    primary = smooth_clip_np((half+ingress-np.abs(orb))/(2.0*ingress))
    secondary = smooth_clip_np((half+ingress-np.abs(sec))/(2.0*ingress))
    return baseline-depth*primary-depth*ratio*secondary + ellip*np.cos(4*np.pi*orb) + reflection*np.cos(2*np.pi*orb)

bounds = [(0.95, 1.05), (0.0, 1.0), (0.002, 0.45), (0.0, 1.0),
          (0.005, 0.12), (0.0, 0.02), (-0.01, 0.01), (np.log(1e-4), np.log(0.03))]
param_names = ["baseline_flux", "phase", "depth_primary", "depth_ratio", "duration_fraction",
               "ellipsoidal_amplitude", "reflection_amplitude", "log_noise"]

profile_rows, summary_rows, plot_rows = [], [], []
for row in CASE_ROWS:
    start_time = time.time()
    matching = aliases[aliases.simulation_row.astype(int).eq(int(row))].sort_values("bic")
    mode = matching.iloc[0]
    center = float(mode.fit_period_days)
    half_width = max(0.05, 0.01*center)
    p_low, p_high = max(0.1, center-half_width), min(13.0, center+half_width)
    # 25 points resolve local shape while keeping this a quick diagnostic.
    period_grid = np.linspace(p_low, p_high, 25)
    t = np.asarray(time_days, float)
    y = np.asarray(observations[int(row)], float)
    valid = np.isfinite(t) & np.isfinite(y)
    t, y = t[valid], y[valid]
    order = np.argsort(t)
    t, y = t[order], y[order]
    t = t-t.min()
    sub = map_params[map_params.simulation_row.astype(int).eq(int(row))]
    map_values = sub.set_index("parameter").map_estimate.astype(float).to_dict()
    truth = parameters.iloc[int(row)]
    fallback = {"baseline_flux": float(np.nanpercentile(y, 90)),
                "phase": float(mode.phase), "depth_primary": 0.1, "depth_ratio": 0.5,
                "duration_fraction": float(np.clip(mode.duration_days/center, 0.0051, 0.1199)),
                "ellipsoidal_amplitude": 0.005, "reflection_amplitude": 0.0,
                "noise_sigma": float(mode.noise_sigma)}
    seed = [map_values.get("baseline_flux", fallback["baseline_flux"]),
            map_values.get("phase", fallback["phase"]),
            map_values.get("depth_primary", fallback["depth_primary"]),
            map_values.get("depth_ratio", fallback["depth_ratio"]),
            map_values.get("duration_fraction", fallback["duration_fraction"]),
            map_values.get("ellipsoidal_amplitude", fallback["ellipsoidal_amplitude"]),
            map_values.get("reflection_amplitude", fallback["reflection_amplitude"]),
            np.log(map_values.get("noise_sigma", fallback["noise_sigma"]))]
    seed = np.array([np.clip(v, lo+1e-8, hi-1e-8) for v,(lo,hi) in zip(seed,bounds)], dtype=float)

    def make_nll(period):
        def nll(theta):
            pred = predict_np(t, theta[0], period, theta[1], theta[2], theta[3], theta[4], theta[5], theta[6])
            log_noise = theta[7]
            return float(0.5*np.sum(((y-pred)/np.exp(log_noise))**2 + 2*log_noise + np.log(2*np.pi)))
        return nll

    local_points = []
    for period in period_grid:
        nll = make_nll(float(period))
        starts = [seed.copy(), np.array(seed, copy=True)]
        starts[1][1] = (starts[1][1]+0.5) % 1.0
        fits = [minimize(nll, x0, method="L-BFGS-B", bounds=bounds,
                         options={"maxiter": 700, "ftol": 1e-9}) for x0 in starts]
        fit = min(fits, key=lambda z: z.fun if np.isfinite(z.fun) else np.inf)
        local_points.append({"simulation_row": int(row), "period_days": float(period),
            "profile_nll": float(fit.fun), "optimizer_success": bool(fit.success),
            "optimizer_message": str(fit.message)})
    minimum_nll = min(x["profile_nll"] for x in local_points)
    for item in local_points:
        item["delta_profile_nll"] = float(item["profile_nll"]-minimum_nll)
        profile_rows.append(item)
    best = min(local_points, key=lambda x:x["profile_nll"])
    tol = max(1e-10, (p_high-p_low)/len(period_grid)*0.51)
    summary_rows.append({"simulation_row": int(row), "selected_bic_period_days": center,
        "profile_min_period_days": best["period_days"], "profile_minimum_at_grid_edge": bool(abs(best["period_days"]-p_low)<=tol or abs(best["period_days"]-p_high)<=tol),
        "profile_lower_edge_delta_nll": float(local_points[0]["profile_nll"]-minimum_nll),
        "profile_upper_edge_delta_nll": float(local_points[-1]["profile_nll"]-minimum_nll),
        "grid_low_days": p_low, "grid_high_days": p_high, "grid_points": len(period_grid),
        "known_period_days_evaluation_only": float(truth["period_days"]),
        "profile_min_period_error_evaluation_only": float(best["period_days"]-float(truth["period_days"])),
        "profile_optimizer_failures": int(sum(not x["optimizer_success"] for x in local_points)),
        "seconds": round(time.time()-start_time, 1)})
    plot_rows.append((int(row), np.array([x["period_days"] for x in local_points]),
                      np.array([x["delta_profile_nll"] for x in local_points]), center))
    pd.DataFrame(profile_rows).to_csv(OUT/"day9_10_period_profile_grid.csv", index=False)
    pd.DataFrame(summary_rows).to_csv(OUT/"day9_10_period_profile_summary.csv", index=False)
    print(f"Row {row}: profile minimum={best['period_days']:.7f} d | at edge={summary_rows[-1]['profile_minimum_at_grid_edge']} | {summary_rows[-1]['seconds']} s", flush=True)

In [ ]:
# Cell 3 — plot profiles and save a diagnostic summary.
ncols = 2
nrows = int(np.ceil(len(plot_rows)/ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4*nrows), squeeze=False, constrained_layout=True)
for ax, (row, periods, delta_nll, center) in zip(axes.flat, plot_rows):
    ax.plot(periods, delta_nll, marker="o", ms=3, lw=1)
    ax.axvline(center, color="darkorange", ls="--", lw=1, label="Day 9.3 lowest-BIC mode")
    ax.set_title(f"Row {row} | profiled period likelihood")
    ax.set_xlabel("Period (days)")
    ax.set_ylabel("Δ profile negative log likelihood")
    ax.legend(fontsize=8)
for ax in axes.flat[len(plot_rows):]:
    ax.set_visible(False)
fig.savefig(OUT/"day9_10_period_profile_diagnostics.png", dpi=150)
plt.show()

summary_df = pd.DataFrame(summary_rows)
summary = {"created_utc": datetime.now(timezone.utc).isoformat(),
    "day7_binary_input": str(BINARY_DIR), "day9_3_alias_input": str(ALIAS_PATH),
    "day9_9_input": str(NUTS_PATH.parent), "profiled_rows": CASE_ROWS,
    "rows_with_profile_minimum_at_grid_edge": summary_df.loc[summary_df.profile_minimum_at_grid_edge, "simulation_row"].astype(int).tolist(),
    "rows_with_profile_optimizer_failures": summary_df.loc[summary_df.profile_optimizer_failures.gt(0), "simulation_row"].astype(int).tolist(),
    "interpretation_limits": [
        "This is a local profile-likelihood diagnostic, not a posterior distribution or confidence interval.",
        "Each curve conditions on one lowest-BIC Day 9.3 mode; other aliases are not marginalized.",
        "Known truth columns are evaluation-only and are not used in optimization.",
        "The six rows are selected difficult cases, not an IID validation sample."
    ],
    "outputs": ["day9_10_period_profile_grid.csv", "day9_10_period_profile_summary.csv", "day9_10_period_profile_diagnostics.png"]}
(OUT/"day9_10_profile_diagnostic_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(summary_df.to_string(index=False))
print(json.dumps(summary, indent=2))
print("Outputs:", OUT)